# Faster R-CNN: two-stage parameter experiments (Kaggle, ~3.5 h GPU)

All runs share one short schedule (`EPOCHS`, `SUBSET_STEP`, `BATCH`, input size); each changes **one**
parameter from the baseline (`--rpn-proposals 2000`, `--roi-pos-fraction 0.25`, `--box-loss smooth_l1`):

| Run | Change | Parameter | Cost |
|---|---|---|---|
| `base` | none | baseline | 1 training run |
| `giou` | `--box-loss giou` | box regression loss (stage 2) | 1 training run |
| `pos05` | `--roi-pos-fraction 0.5` | RoI sampling ratio (stage 2), optional | 1 training run, only if time is left |
| `base` @ 300 / 1000 proposals | test-time RPN proposals | number of RPN proposals (stage 1) | eval only |

Before running: **Settings -> Accelerator -> GPU**, **Internet on**, add the competition data
(**Add Input -> multiple-object-video-detection-hw-2-ucu-ai-2026**).
Use **Save Version -> Save & Run All** so `/kaggle/working/runs` survives the session.
Each run is evaluated right after training, so a timeout loses at most the run in progress.

In [ ]:
!nvidia-smi
!git clone -q https://github.com/LiliAlania/drone-multi-object-detection.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pycocotools

In [ ]:
import sys, time
from pathlib import Path

T0 = time.time()
REPO = Path("/kaggle/working/repo")
sys.path.insert(0, str(REPO))
MOD = "src.models.faster_rcnn"
assert (REPO / "src/models/faster_rcnn.py").is_file(), "push the src/models/ layout to GitHub first"

EXPERIMENTS = {
    "base": "",
    "giou": "--box-loss giou",
    "pos05": "--roi-pos-fraction 0.5",  # runs only if the time budget allows
}
RUN = ["base", "giou", "pos05"]
PROPOSALS = [300, 1000]  # test-time sweep on `base`; its own eval is the 2000 point

EPOCHS = 5         # step LR drops after epochs 3 and 4; lower it if one epoch takes > 9 min
SUBSET_STEP = 20   # every 20th train frame: ~1.2k images, one epoch ~8 min on a T4
BATCH = 4          # 2 if CUDA out of memory
EVAL_STEP = 5      # every 5th val frame, as in the README
BUDGET_MIN = 210   # whole session; keep under the GPU time you have
RESERVE_MIN = 20   # kept free for the proposals sweep and the summary
RUNS = Path("/kaggle/working/runs")

In [ ]:
# find the competition split folders without walking every image
INPUT = Path("/kaggle/input")
found = [p for depth in range(1, 5) for p in INPUT.glob("*/" * depth + "val/annotations_coco.json")]
assert found, "Competition data not found: add it with Add Input"
VAL_ROOT = found[0].parent
DATA = VAL_ROOT.parent
print("data:", DATA, sorted(p.name for p in DATA.iterdir()))

SUBSET = Path("/kaggle/working/visdrone-subset")
if not (SUBSET / "images" / "train").is_dir():
    !python -m src.make_subset --data-root {DATA} --output-dir {SUBSET} --step {SUBSET_STEP} --splits train val
    !rm -f {SUBSET}.zip
print(f"{len(list((SUBSET / 'images' / 'train').rglob('*.jpg')))} train images, "
      f"{(time.time() - T0) / 60:.0f} min elapsed")

In [ ]:
run_minutes = []  # wall time of finished runs, to decide whether the next one still fits
done = []
for name in RUN:
    elapsed = (time.time() - T0) / 60
    if run_minutes and elapsed + max(run_minutes) > BUDGET_MIN - RESERVE_MIN:
        print(f"skip {name}: {elapsed:.0f} min used, a run takes ~{max(run_minutes):.0f} min")
        continue
    flags = EXPERIMENTS[name]
    out = RUNS / f"frcnn_{name}"
    resume = "--resume" if (out / "last.pt").is_file() else ""  # rerunning this cell continues an interrupted run
    print(f"=== {name}: {flags or 'baseline'} ===")
    start = time.time()
    !python -m {MOD} train --data-root {SUBSET} --epochs {EPOCHS} --batch {BATCH} --workers 4 {flags} --out-dir {out} {resume}
    !python -m {MOD} evaluate --weights {out}/last.pt --split-root {VAL_ROOT} --batch {BATCH} --step {EVAL_STEP} | tee {out}/eval.txt
    run_minutes.append((time.time() - start) / 60)
    done.append(name)
    print(f"{name}: {run_minutes[-1]:.0f} min, {(time.time() - T0) / 60:.0f} min elapsed")

In [ ]:
# RPN proposals at test time: same `base` weights, only the number of proposals passed to the RoI head changes
import importlib
import torch

frcnn = importlib.import_module(MOD)

def with_proposals(n):
    # the checkpoint's saved model_cfg holds rpn_proposals=2000; override it after it is passed in
    def build(*args, **kwargs):
        return frcnn.build_model(*args, **{**kwargs, "rpn_proposals": n})
    return build

sweep = []
for n in PROPOSALS:
    print(f"=== base @ {n} proposals ===")
    metrics = frcnn.evaluate(RUNS / "frcnn_base" / "last.pt", VAL_ROOT, BATCH, EVAL_STEP, build_fn=with_proposals(n))
    sweep.append({"run": "base", "change": f"{n} RPN proposals (test)", "rpn_proposals": n, **metrics})
    torch.cuda.empty_cache()

In [ ]:
# summary: mAP, AP50, AP_small and inference speed per run
import re
import pandas as pd

rows = []
for name in done:
    text = (RUNS / f"frcnn_{name}" / "eval.txt").read_text(errors="ignore")
    ap = re.search(r"mAP@\[\.5:\.95\] ([\d.]+) \| AP50 ([\d.]+) \| AP_small ([\d.]+)", text)
    ms = re.search(r"Inference: ([\d.]+) ms/image", text)
    if not ap:
        print(f"{name}: no metrics in eval.txt")
        continue
    rows.append({"run": name, "change": EXPERIMENTS[name] or "baseline", "rpn_proposals": 2000,
                 "mAP": float(ap[1]), "AP50": float(ap[2]), "AP_small": float(ap[3]),
                 "ms_per_image": float(ms[1]) if ms else None})

cols = ["run", "change", "rpn_proposals", "mAP", "AP50", "AP_small", "ms_per_image"]
summary = pd.DataFrame(rows + sweep)[cols]
summary.to_csv(RUNS / "frcnn_experiments.csv", index=False)
print(f"settings: epochs {EPOCHS}, subset step {SUBSET_STEP}, batch {BATCH}, eval step {EVAL_STEP}")
summary.round(4)

In [ ]:
# keep only checkpoints, eval logs and the summary CSV in the saved output
!rm -rf {SUBSET} /kaggle/working/repo